# Clustering delle regioni per profilo di mercato del lavoro

Segmentazione delle regioni italiane su più indicatori (disoccupazione, occupazione, inattività, NEET) con K-means, con visualizzazione via PCA.

**Nota sui dati**: nei dati Eurostat ogni indicatore ha una fascia d'età diversa (disoccupazione 15-74, occupazione/inattività 20-64/15-64, NEET 15-29) — è la definizione standard di ciascun indicatore, non un problema: qui non filtriamo per classe d'età, ogni indicatore porta semplicemente la propria.

**Prerequisito**: il DAG deve aver completato almeno un run con successo.

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import plotly.express as px
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
from sqlalchemy import create_engine

from src.config import warehouse_connection_string

engine = create_engine(warehouse_connection_string())

## Costruzione della matrice regioni x indicatori

Media sull'ultimo anno disponibile, per indicatore e per regione (sesso=totale, classe_eta=totale).

In [2]:
fact = pd.read_sql('SELECT * FROM gold.fact_mercato_lavoro', engine)
dim_territorio = pd.read_sql('SELECT * FROM gold.dim_territorio', engine)
dim_tempo = pd.read_sql('SELECT * FROM gold.dim_tempo', engine)

df = fact.merge(dim_territorio, on='cod_territorio').merge(dim_tempo, on='periodo')
df = df[df['sesso'] == 'T']  # 'T' = totale maschi+femmine

ultimo_anno = df['anno'].max()
recente = df[df['anno'] == ultimo_anno]

matrice = recente.pivot_table(index='territorio', columns='indicatore', values='valore', aggfunc='mean')
matrice = matrice.dropna()
print(f'Regioni: {len(matrice)}  |  Indicatori disponibili: {list(matrice.columns)}')
matrice.head()

Regioni: 19  |  Indicatori disponibili: ['neet', 'tasso_disoccupazione', 'tasso_inattivita', 'tasso_occupazione']


indicatore,neet,tasso_disoccupazione,tasso_inattivita,tasso_occupazione
territorio,,,,
Abruzzo,14.0,6.5,33.0,67.2
Basilicata,14.9,5.3,39.3,61.8
Calabria,20.9,9.8,48.3,50.3
Campania,21.5,13.9,45.6,50.8
Emilia-Romagna,8.2,4.1,25.5,77.0


In [3]:
scaler = StandardScaler()
X = scaler.fit_transform(matrice.values)

K = min(4, len(matrice) - 1) if len(matrice) > 2 else 2
kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
matrice['cluster'] = kmeans.fit_predict(X)

if len(matrice) > K:
    print('Silhouette score:', silhouette_score(X, matrice['cluster']))

Silhouette score: 0.41483434035784555


## Visualizzazione via PCA (2 componenti)

In [4]:
pca = PCA(n_components=2)
coords = pca.fit_transform(X)
plot_df = matrice.reset_index()[['territorio', 'cluster']].copy()
plot_df['pc1'], plot_df['pc2'] = coords[:, 0], coords[:, 1]

px.scatter(
    plot_df, x='pc1', y='pc2', color=plot_df['cluster'].astype(str), text='territorio',
    title=f'Cluster regionali per profilo di mercato del lavoro ({ultimo_anno})'
)

## Interpretazione dei cluster: media indicatori per gruppo

In [5]:
matrice.groupby('cluster').mean(numeric_only=True)

indicatore,neet,tasso_disoccupazione,tasso_inattivita,tasso_occupazione
cluster,,,,
0,14.450000,6.825,36.5250,63.400
1,21.050000,11.475,45.7500,51.925
2,8.431250,4.000,26.8875,75.650
3,12.833333,5.100,29.5000,72.000


## (Opzionale) Salvare l'assegnazione dei cluster in `gold.cluster_regioni`

per poterla mostrare nella dashboard senza rieseguire il notebook.

In [6]:
out = matrice.reset_index()[['territorio', 'cluster']]
out['anno_riferimento'] = ultimo_anno

# out.to_sql('cluster_regioni', engine, schema='gold', if_exists='replace', index=False)
out

indicatore,territorio,cluster,anno_riferimento
0,Abruzzo,0,2025
1,Basilicata,0,2025
2,Calabria,1,2025
3,Campania,1,2025
4,Emilia-Romagna,2,2025
5,Friuli-Venezia Giulia,2,2025
6,Lazio,3,2025
7,Liguria,3,2025
8,Lombardia,2,2025
9,Marche,3,2025
